# EXP-012 — Colab GPU (완료 2026-07-12)

공식: SillokBERT full-data fine-tune + valid_loss best 선택 + test 9,719.
결과: Exact@1 33.75% / @5 53.04% / @10 60.47% · DECISIONS [010]
로컬 백업: `notebooks/best/` · 스냅샷: `reports/exp012_results.md`
상세 재현: `reports/colab_exp012_runbook.md`

주의:
- smoke 결과는 공식 성능이 아님. EXP-011 경로를 덮어쓰지 말 것.
- 기본 출력은 Drive가 아니라 `/content/exp012/` (Drive 용량 초과 방지).
- private 레포이므로 **브랜치 지정 + GitHub 토큰**으로 clone.
- `rm -rf /content/sooyah` 전에 반드시 `os.chdir("/content")` (삭제된 cwd 에러 방지).

## 0a) Repo clone (토큰 + 브랜치)

런타임을 새로 시작했거나 `/content/sooyah`가 없을 때만 실행.

In [ ]:
import os
from getpass import getpass

# 이전 셀에서 /content/sooyah 로 chdir 한 뒤 rm 하면 cwd 소실 에러가 남
os.chdir("/content")
token = getpass("GitHub token: ")

!rm -rf /content/sooyah
!git clone -b cursor/cloud-agent-1783706016574-58a09 https://{token}@github.com/google-ajou-Epigraph/sooyah.git /content/sooyah
!ls /content/sooyah

os.chdir("/content/sooyah")
!python -m src.data.finetune_sillokbert --help | head -n 40

## 0b) Paths + cwd

학습·평가는 `/content`에 저장. Drive 마운트는 마지막 반출 셀에서만.

In [ ]:
import os

REPO = "/content/sooyah"
OUT = "/content/exp012/sillokbert_exp012_gpu"
SMOKE_OUT = "/content/exp012/_exp012_smoke"
PRED_CSV = "/content/exp012/exp012_sillok_ft_full_predictions.csv"
SUMMARY_CSV = "/content/exp012/evaluation_summary.csv"

os.makedirs(OUT, exist_ok=True)
os.makedirs(SMOKE_OUT, exist_ok=True)
os.chdir(REPO)
print("cwd", os.getcwd())
print("OUT", OUT)

In [ ]:
!python -m pip -q install -r requirements.txt
!python work/check_cuda_env.py
!python work/colab_preflight.py

## 1) Smoke (비공식)

In [ ]:
!python -m src.data.finetune_sillokbert \
  --run-mode smoke \
  --planned-experiment-id EXP-012 \
  --output-dir {SMOKE_OUT} \
  --epochs 1 \
  --batch-size 8

## 2) Full-train

In [ ]:
!python -m src.data.finetune_sillokbert \
  --run-mode full \
  --planned-experiment-id EXP-012 \
  --output-dir {OUT} \
  --epochs 2 \
  --batch-size 16 \
  --learning-rate 5e-5

## 2b) Resume (세션 끊김 시 — checkpoint-epoch-N 경로 수정)

In [ ]:
# 예: epoch 1 완료 후 재개
# !python -m src.data.finetune_sillokbert \
#   --run-mode full \
#   --planned-experiment-id EXP-012 \
#   --output-dir {OUT} \
#   --epochs 2 \
#   --batch-size 16 \
#   --learning-rate 5e-5 \
#   --resume-from {OUT}/checkpoint-epoch-1

## 3) Best 확인

In [ ]:
from pathlib import Path
print(Path(OUT, "best_checkpoint.json").read_text(encoding="utf-8"))

## 4) Full test (best만)

In [ ]:
!python -m src.eval.run_fill_mask_eval \
  --run-mode full \
  --experiment-id EXP-012 \
  --experiment-name "SillokBERT GPU full-data fine-tuned full test" \
  --model-kind ddokbaro/SillokBert \
  --model-path {OUT}/best \
  --require-model-path \
  --split test \
  --batch-size 32 \
  --overwrite \
  --predictions-csv {PRED_CSV} \
  --summary-csv {SUMMARY_CSV} \
  --notes "planned EXP-012; Colab GPU; model_path=best by valid_loss; full test n=9719"

## 5) (선택) Drive로 결과만 반출

Drive 여유 공간이 있을 때만. `best/` + JSON + CSV만 복사 (중간 checkpoint는 제외).

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/exp012_export")
dest.mkdir(parents=True, exist_ok=True)

for name in ("best", "best_checkpoint.json"):
    src = Path(OUT) / name
    if not src.exists():
        print("skip missing", src)
        continue
    target = dest / name
    if src.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(src, target)
    else:
        shutil.copy2(src, target)
    print("copied", src, "->", target)

for src in (PRED_CSV, SUMMARY_CSV):
    p = Path(src)
    if p.exists():
        shutil.copy2(p, dest / p.name)
        print("copied", p, "->", dest / p.name)
    else:
        print("skip missing", p)